# 非平衡样本的处理——SMOTE过采样

由描述性统计结果可以看出，数据集存在比较多的缺失值，本研究采取剔除所有含有缺失值的的样本的处理方法。

变量“com_endow”的平均值为0.0999，表明被解释变量存在严重的不平衡问题，绝大多数样本未参与商业养老保险。这会导致训练集中阳性样本数量严重不足，机器学习模型无法通过分析充足的阳性样本以捕捉变量间的隐含关系，无法达到理想的训练效果，因此本研究采用smote过采样技术对训练集中的阳性样本进行扩充。

首先利用分层抽样法将剔除缺失值后的样本划分为训练集和测试集，其中70%为训练集，30%为测试集。训练集中共有5725个阴性样本，444个阳性样本，利用smote过采样法在每个阳性样本周围选择5个最近邻进行连线，并随机在这些连线上进行线性插值。为了让阳性样本特征得到充分学习，本文选择将阳性样本过采样到与阴性样本数量相同，最终得到5725个阴性样本，5725个阳性样本，共计11450个样本。

第一步：导入数据(from 2_3)

In [5]:
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

In [ ]:
df_2years = pd.read_excel("两年清洗后数据.xlsx", index_col = 0)
print(df_2years.head())

第二步：筛选需要的变量

In [9]:
df_2years_part = df_2years[['com_endow', 'age', 'marriage', 'member', 'edu','son',
                        'daughter','house', 'media_old', 'media_new', 'social','urban',
                        'family_income', 'happy', 'depression', 'self_health',
                        'basic_endow', 'basic_medicine', 'opinion_gov',
                        'opinion_kid', 'opinion_self', 'opinion_all','investment']]

第三步：剔除缺失值

In [10]:
df_2years_nona = df_2years_part.dropna()

第四步：划分训练集和测试集（70%：30%）

In [12]:
X = df_2years_nona.drop('com_endow', axis = 1)
y = df_2years_nona['com_endow']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=22, stratify=y)

第五步：对测试集进行smote过采样，缓解不平衡样本的影响。

In [18]:
sampling_strategy = 1

smote = SMOTE(sampling_strategy = 1, random_state=77)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

resampled_df_2years = pd.DataFrame(X_resampled, columns=X.columns)
resampled_df_2years['com_endow'] = y_resampled

print('original:', y_train.value_counts())
print('resampled:', y_resampled.value_counts())

original: com_endow
0.0    5725
1.0     444
Name: count, dtype: int64
resampled: com_endow
0.0    5725
1.0    5725
Name: count, dtype: int64


通过smote过采样法将阳性样本采样到与阴性样本数量相等（5725:5725）

第六步：构造测试集

In [19]:
columns = ['age', 'marriage', 'member', 'edu','son',
                        'daughter','house', 'media_old', 'media_new', 'social','urban',
                        'family_income', 'happy', 'depression', 'self_health',
                        'basic_endow', 'basic_medicine', 'opinion_gov',
                        'opinion_kid', 'opinion_self', 'opinion_all','investment',
                        'com_endow']
test_data = np.column_stack((X_test, y_test))  # 将特征和标签水平合并
test_df = pd.DataFrame(test_data, columns=columns)

第七步：导出数据

In [ ]:
resampled_df_2years.to_excel("两年训练集过采样.xlsx", index = False) 
test_df.to_excel("两年测试集.xlsx", index = False) 
